# NEXUS CoboSense — LSTM Intent Classifier Training
**Run on Google Colab Free GPU**
Generates synthetic pose sequences → trains LSTM → saves model.pt

In [ ]:
# Install deps
!pip install torch scikit-learn numpy --quiet


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

# ── Config ──────────────────────────────────────
SEQ_LEN    = 10
N_FEATURES = 66   # 33 landmarks × (x, y)
N_SAMPLES  = 50000
N_CLASSES  = 3
# 0=static  1=leaving  2=approaching


In [ ]:
# ── Synthetic data generator ─────────────────────
np.random.seed(42)

def gen_static(n):
    '''Person standing still — landmarks jitter slightly'''
    base = np.random.uniform(0.3, 0.7, (n, N_FEATURES))
    seqs = [base + np.random.normal(0, 0.005, (n, N_FEATURES))
            for _ in range(SEQ_LEN)]
    return np.stack(seqs, axis=1)  # (n, 10, 66)

def gen_leaving(n):
    '''Person moving away — landmarks shrink toward center'''
    base = np.random.uniform(0.3, 0.7, (n, N_FEATURES))
    seqs = []
    for t in range(SEQ_LEN):
        scale = 1.0 - t * 0.015
        center = 0.5
        shifted = center + (base - center) * scale
        shifted += np.random.normal(0, 0.008, (n, N_FEATURES))
        seqs.append(shifted)
    return np.stack(seqs, axis=1)

def gen_approaching(n):
    '''Person moving toward camera — landmarks expand'''
    base = np.random.uniform(0.3, 0.7, (n, N_FEATURES))
    seqs = []
    for t in range(SEQ_LEN):
        scale = 1.0 + t * 0.02
        center = 0.5
        shifted = center + (base - center) * scale
        shifted = np.clip(shifted, 0, 1)
        shifted += np.random.normal(0, 0.008, (n, N_FEATURES))
        seqs.append(shifted)
    return np.stack(seqs, axis=1)

n_each = N_SAMPLES // 3
X0 = gen_static(n_each);      y0 = np.zeros(n_each, dtype=int)
X1 = gen_leaving(n_each);     y1 = np.ones(n_each, dtype=int)
X2 = gen_approaching(n_each); y2 = np.full(n_each, 2, dtype=int)

X = np.vstack([X0, X1, X2])
y = np.concatenate([y0, y1, y2])
print(f'Dataset: {X.shape}, labels: {np.bincount(y)}')


In [ ]:
# ── Scale + split ─────────────────────────────────
X_flat = X.reshape(-1, N_FEATURES)
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(-1, SEQ_LEN, N_FEATURES)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_tr.shape}  Val: {X_val.shape}')


In [ ]:
# ── Model ─────────────────────────────────────────
class CoboLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(N_FEATURES, 64, batch_first=True)
        self.drop1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(64, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.2)
        self.fc    = nn.Linear(32, N_CLASSES)
    def forward(self, x):
        o, _ = self.lstm1(x)
        o = self.drop1(o)
        o, _ = self.lstm2(o)
        o = self.drop2(o)
        return self.fc(o[:, -1, :])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')
model = CoboLSTM().to(device)


In [ ]:
# ── Train ─────────────────────────────────────────
EPOCHS = 30
BATCH  = 256

Xtr = torch.tensor(X_tr, dtype=torch.float32)
ytr = torch.tensor(y_tr, dtype=torch.long)
Xvl = torch.tensor(X_val, dtype=torch.float32)
yvl = torch.tensor(y_val, dtype=torch.long)

loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    model.eval()
    with torch.no_grad():
        val_pred = model(Xvl.to(device)).argmax(1)
        val_acc  = (val_pred == yvl.to(device)).float().mean().item()
    avg_loss = total_loss / len(loader)
    sched.step(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f'Ep {epoch+1:2d}/{EPOCHS} | loss={avg_loss:.4f} | val_acc={val_acc:.4f}')

print('Training complete!')


In [ ]:
# ── Save ──────────────────────────────────────────
import os
os.makedirs('models/lstm_cobosense', exist_ok=True)

torch.save(model.cpu().state_dict(), 'models/lstm_cobosense/model.pt')
with open('models/lstm_cobosense/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved: models/lstm_cobosense/model.pt')
print('Saved: models/lstm_cobosense/scaler.pkl')
print('Download these files and place in your nexus/ project.')
